# Unitary ESPRIT and Forward-Backward Averaging 

In [ ]:
# Importing modules
import numpy as np;

# np.random.seed(42); # In order to ensure reproducibility. You can use different seed values.

In [ ]:
# Defining methods
hermitian = lambda array: np.conj(array).T; 


def steering_vector(sensor_pos: np.ndarray, ang_elev: float, ang_azim: float, wavelen: float) -> np.ndarray:
  omega = np.array([[np.sin(ang_elev)*np.cos(ang_azim)],
                    [np.sin(ang_elev)*np.sin(ang_azim)],
                    [np.cos(ang_elev)]]); 
  return np.exp(1j*2*np.pi/wavelen*sensor_pos.T@omega); 


def d_steering_vector(sensor_pos: np.ndarray, ang_elev: float, ang_azim: float, wavelen: float) -> np.ndarray:
  a = steering_vector(sensor_pos, ang_elev, ang_azim, wavelen); 
  d_omega = np.array([[np.cos(ang_elev)*np.cos(ang_azim)],
                      [np.cos(ang_elev)*np.sin(ang_azim)],
                      [-np.sin(ang_elev)]]); 
  return -1j*2*np.pi/wavelen * a * sensor_pos.T@d_omega; 


def generate_pos_1d_ula(N: int, d: float, axis=(1.,0.,0.), x_init=(0.,0.,0.)) -> np.ndarray:
  if len(axis) != 3:
    raise TypeError(f"""The axis argument represents a 3D Cartesian vector.
      The length of the input ({len(axis)}) does not match with expected
      size 3."""); 
  if sum(axis) != 1:
    axis_new = (x/sum(axis) for x in axis); 
    axis = axis_new; 

  if len(x_init) != 3:
    raise TypeError(f"""The x_init argument represents a 3D cartesian vector.
      The length of the input ({len(x_init)}) does not match with expected
      size 3."""); 

  sensor_pos = np.tile(x_init, N).reshape(N,3).T \
    + (np.arange(0,N,1)*d) * np.tile(np.array(axis), N).reshape(N,3).T; 
  return sensor_pos; 


def calculate_crb(sensor_pos: np.ndarray, N: int, T: int, wl: float, angs_elev: np.ndarray, snr_db: float, S_db: list) -> np.ndarray:
  noise_pow = 10**(-snr_db/10); 
  sig_pow = [10**(s_db)/10 for s_db in S_db]; 
  R_ss = np.diag(np.array(sig_pow)); 

  K = angs_elev.shape[0]; 

  # Steering matrix (A_mat) Derivative matrix (D_mat) of steering vectors w.r.t angles (in degrees)
  A_mat = np.zeros((N, K), dtype=complex); 
  D_mat = np.zeros((N, K), dtype=complex); 

  for i, theta in enumerate(angs_elev):
    A_mat[:,i] = steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl)[:,0]; 
    D_mat[:,i] = d_steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl)[:,0]; 

  Rxx_true = A_mat@R_ss@A_mat.conj().T + noise_pow*np.eye(N); 
  Rxx_inv = np.linalg.inv(Rxx_true); 

  P_A_perp = np.eye(N) - A_mat@np.linalg.inv(A_mat.conj().T@A_mat)@A_mat.conj().T; # Projection matrix onto the noise subspace

  # The Fisher Information Matrix components
  term1 = D_mat.conj().T @ P_A_perp @ D_mat; 
  term2 = (R_ss @ A_mat.conj().T @ Rxx_inv @ A_mat @ R_ss).T; 

  FIM = (2*T/noise_pow) * np.real(term1*term2); 

  return np.linalg.inv(FIM); 


def generate_random_targets(ang_min: float, ang_max: float, ang_dist: float, K: int) -> np.ndarray:
  while True:
    angs = np.random.uniform(ang_min, ang_max, K); 
    angs_sort = np.sort(angs); 
    for i in range(K-1):  # break for if is not valid
      is_valid = np.abs(angs_sort[i+1] - angs_sort[i]) >= ang_dist; 
      if not is_valid:
        break; 
    if is_valid: break;   # break while if is valid
  return angs; 

## Benchmark Methods: Root-MUSIC, ESPRIT

In [ ]:
def doa_est_esprit(R_xx: np.ndarray, d: float, wl: float, K: int) -> np.ndarray:
  U, _, _ = np.linalg.svd(R_xx);                    # Left singular matrix
  Us = U[:,:K];                                     # Signal Subspace

  U1 = Us[:-1,:];                                   # First M-1 rows (Subarray 1)
  U2 = Us[1:,:];                                    # Last M-1 rows (Subarray 2)
  U12 = np.hstack((U1, U2)); 

  _, _, Vh = np.linalg.svd(U12); 
  Vh = Vh.T.conj(); 
  V12 = Vh[:K,K:]; 
  V22 = Vh[K:,K:]; 

  Psi = -V12 @ np.linalg.inv(V22);                  # Rotation matrix

  eig_vals = np.linalg.eigvals(Psi);                # Complex eigenvalues of the rotation matrix
  phases = np.angle(eig_vals);                      # The phases (angles) of the eigenvalues

  estimated_angles_rad = np.arcsin(phases/(2*np.pi*(d/wl))); 
  return np.sort(np.rad2deg(estimated_angles_rad)); # returns in degrees


def doa_est_root_music(R_xx: np.ndarray, wl: float, d: float, N:int, K: int):
  _, e_vec = np.linalg.eigh(R_xx); 
  Un = e_vec[:, :-K]; 
  C = Un @ hermitian(Un); 

  coeffs = np.zeros(2*N-1, dtype=complex); # The polynomial coefficients
  for i, k in enumerate(range(N-1,-N,-1)):
    coeffs[i] = np.sum(np.diag(C, k)); 
  
  roots = np.roots(coeffs); # The roots of the polynomial coefficients
  root_i = roots[np.abs(roots) < 1.0]; # The roots inside the unit circle (|z| < 1)
  root_i_sort = root_i[np.argsort(np.abs(root_i))[::-1]]; 
  root_i_sel = root_i_sort[:K]; 

  # Extracting phases and calculating the corresponding angles
  phases = np.angle(root_i_sel); 
  sin_theta = (phases*wl)/(2*np.pi*d); 

  # Clip values to handle occasional minor numerical precision edge cases outside [-1, 1]
  sin_theta = np.clip(sin_theta, -1.0, 1.0); 

  # Convert to degrees and sort
  est_angles = np.sort(np.rad2deg(np.arcsin(sin_theta))); 

  return est_angles; 

In [ ]:
# Defining parameters
c = 3*1e8;      # The speed of light in vacuum (m/s)
f_c = 5*1e9;    # Narrowband signal frequency (Hz)
f_s = 100*1e9;  # Receiver sampling frequency (Hz)
wl = c/f_c;     # Signal wavelenght (m)
d = wl/2;       # Antenna distance (m)

N = 16;         # Number of antennas
T = 1000;       # Number of snapshots
K = 5;          # Number of targets

S_db = [0]*K;   # Target signal power (dB)
snr_db = 10.0;  # Signal-to-Noise Ratio (dB)

ang_azim = 0;   # Azimuth angle (deg)
ang_min = -60;  # Minimum elevation angle (deg)
ang_max = 60;   # Maximum elevation angle (deg)
ang_dist = 5;   # Minimum distance between targets (deg)
ang_res = 0.1   # Angular resolution in scanning (deg)

sensor_pos = generate_pos_1d_ula(N, d);                                 # Sensor position
theta_scan = np.arange(-90,90+ang_res,ang_res);                         # Angle scan
true_angles = generate_random_targets(ang_min, ang_max, ang_dist, K);   # Target elevation angles

for k in range(K):
  print(f"Target {k} True Angle: {true_angles[k]:.3f}°"); 

In [ ]:
# Data Generation
A = np.column_stack([steering_vector(sensor_pos, np.deg2rad(theta), np.deg2rad(0), wl) for theta in true_angles]); # Array Steering Matrix
S_amp = np.diag([np.sqrt(10**(s_db / 10)) for s_db in S_db]); 
S = S_amp @ (np.random.randn(K, T) + 1j*np.random.randn(K, T))/np.sqrt(2); # Target Signals: Uncorrelated Gaussian (variance = 1.0)

# Noise: Spatially white complex Gaussian noise
noise_pow = 10**(-snr_db/10); 
Noise = np.sqrt(noise_pow) * (np.random.randn(N, T) + 1j*np.random.randn(N, T))/np.sqrt(2); 

# Received Signal at the Array
X = A @ S + Noise; 

## The ESPRIT Algorithm

In [ ]:
R_xx = (X @ hermitian(X))/T;      # Sample covariance matrix

est_esprit = doa_est_esprit(R_xx, d, wl, K); 
est_rmusic = doa_est_root_music(R_xx, wl, d, N, K); 

## The Unitary ESPRIT Algorithm

In [ ]:
# Helper function for unitary transformation matrix
def generate_Q(M: int) -> np.ndarray:
  """
  Generates the sparse unitary transformation matrix Q_M to be used in the
  Unitary ESPRIT. Works for both even and odd array dimensions.

  --Input--
  M: Array of length 2K. Type of int.

  --Output--
  Q: Unitary transformation matrix. Type of np.ndarray of size (M,M) and dtype of np.complex128.
  """
  K = M // 2; 
  if M % 2 == 0:  # Even number of elements
    I_K = np.eye(K);
    J_K = np.fliplr(I_K);
    Q = (1.0 / np.sqrt(2)) * np.block([
      [I_K,  1j * I_K],
      [J_K, -1j * J_K]
    ]); 
  else:           # Odd number of elements
    I_K = np.eye(K);
    J_K = np.fliplr(I_K);
    zeros_col = np.zeros((K, 1)); 
    zeros_row = np.zeros((1, K)); 
    Q = (1.0 / np.sqrt(2)) * np.block([
      [I_K,       zeros_col,  1j * I_K],
      [zeros_row, np.sqrt(2), zeros_row ],
      [J_K,       zeros_col, -1j * J_K]
    ]); 
  return Q; 

In [ ]:
# The Unitary ESPRIT algorithm

# Forward-Backward Averaging
J_N = np.fliplr(np.eye(N));      # Exchange matrix
R_fb = 0.5 * (R_xx + J_N @ R_xx.conj() @ J_N); 

# Real-valued transformation
Q_N = generate_Q(N); 
# The theoretical result is strictly real, but we use np.real() to discard any tiny imaginary leftovers.
R_real = np.real(hermitian(Q_N) @ R_fb @ Q_N); 

# Real subspace extraction
U, _, _ = np.linalg.svd(R_real); 
Es = U[:,:K];                                   # Real-valued Signal Subspace

# Standard selection matrices for N-1 overlapping subarrays to form the Real-Valued Selection Matrices (K1 and K2)
J1 = np.hstack((np.eye(N-1), np.zeros((N-1, 1)))); 
J2 = np.hstack((np.zeros((N-1, 1)), np.eye(N-1))); 

Q_N_m1 = generate_Q(N-1); 

# Transform into real-valued selection matrices
K1 = np.real(hermitian(Q_N_m1) @ (J1 + J2) @ Q_N); 
K2 = np.real(hermitian(Q_N_m1)@ (1j * (J1 - J2)) @ Q_N); 

# TLS
K1_Es = K1 @ Es; 
K2_Es = K2 @ Es; 
U12 = np.hstack((K1_Es, K2_Es)); 

_, _, Vh_tls = np.linalg.svd(U12); 
V_tls = Vh_tls.T.conj(); 

# Partition the V matrix to isolate the null space
V12 = V_tls[:K,K:]; 
V22 = V_tls[K:,K:]; 

# Calculate the real-valued TLS Rotation Matrix
Upsilon_tls = -V12 @ np.linalg.inv(V22); 

# Extracting angles

omega = np.linalg.eigvals(Upsilon_tls);         # The eigenvalues correspond to omega_i = tan(mu_i / 2)

# Back-calculate the phase shifts (mu_i)
mu = 2 * np.arctan(omega); 

# Back-calculate the spatial angles of arrival
u_esprit_angles_rad = np.arcsin(mu / (2*np.pi*(d/wl))); 
est_u_esprit = np.sort(np.rad2deg(u_esprit_angles_rad)); 

In [ ]:
# Printing Results
true_angles_sorted = true_angles[np.argsort(true_angles)]; 
S_db_sorted = np.array(S_db)[np.argsort(true_angles)]; 

## Cramer-Rao Bound
CRB_mat = calculate_crb(sensor_pos, N, true_angles_sorted, snr_db, S_db_sorted); 
CRB_var = np.diag(CRB_mat);       # The diagonal elements are the minimum variances for target 1 and target 2
CRB_rmse = np.sqrt(CRB_var);      # Convert variance (deg^2) to Standard Deviation / RMSE (deg)

print(f"--- DoA Estimation with the Unitary ESPRIT Algorithm ---"); 
print(f"Array Size:        {N} elements"); 
print(f"Snapshots:         {T}"); 
print(f"Target No.:        {K}"); 
print(f"SNR:               {snr_db} dB\n"); 

print("-" * 90); 
print("Target | CRLB (deg) || True Angle (deg) | ESPRIT Est. (deg) | ESPRIT Err. (deg) || Unitary ESPRIT Est. (deg) | Unitary ESPRIT Err. (deg ) |"); 
print("-" * 90); 
for k in range(K):
  esprit_err = np.abs(true_angles_sorted[k] - est_esprit[k]); 
  u_esprit_err = np.abs(true_angles_sorted[k] - est_u_esprit[k]); 
  print(f"  {k+1}    | {true_angles_sorted[k]:>16.4f} | {est_esprit[k]:>16.4f} | {esprit_err:>16.4f} | {est_u_esprit[k]:>24.4f} | {u_esprit_err:>11.4f} | {CRB_rmse[k]:>10.4f} |"); 


--- DoA Estimation with the Unitary ESPRIT Algorithm ---
Array Size:        16 elements
Snapshots:         100
Target No.:        5
SNR:               10.0 dB

------------------------------------------------------------------------
Target | True Angle (deg) | ESPRIT Est.(deg) | Error (deg) | CRLB (deg) |
------------------------------------------------------------------------
  1    |         -25.9973 |         -26.0112 |      0.0139 |     0.0014 |
  2    |          -1.8519 |          -1.7710 |      0.0808 |     0.0013 |
  3    |          18.3090 |          18.3106 |      0.0015 |     0.0014 |
  4    |          68.0286 |          68.2518 |      0.2232 |     0.0169 |
  5    |          74.0179 |          74.4182 |      0.4003 |     0.0230 |
--------------------------------------------------------------------------------
Target | True Angle (deg) | Unitary ESPRIT Est.(deg) | Error (deg) | CRLB (deg) |
--------------------------------------------------------------------------------
  1   